In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pcmci_sweep as sw

%load_ext autoreload
%autoreload 2
pd.set_option("display.width", 200)


In [ ]:
FREQ  = "10min"
FLOWS = "pos"        # "pos" | "main" | "burner"
RATIO = True

d, flows = sw.load_raw()                     # 30s, cleaned, TARGET_CLEAN built
R = sw.resample(d, flows, FREQ)              # resampled, cleaning-blanked, logged
C = sw.build_frame(R, FLOWS, RATIO)          # parameterized + differenced

ALL      = list(R.columns)
complete = R.notna().all(axis=1)
print(f"{len(R)} rows @ {FREQ} x {len(ALL)} vars | complete {complete.mean():.1%}")
print(f"C: {C.shape[1]} vars -> {list(C.columns)}")


In [ ]:
rep = pd.DataFrame({
    "nan%": (R[ALL].isna().mean() * 100).round(2),
    "std":   R[ALL].std().round(4),
    "min":   R[ALL].min().round(2),
    "max":   R[ALL].max().round(2),
})
rep["verdict"] = np.where(rep["std"].fillna(0) < 1e-9, "CONSTANT -> drop",
                  np.where(rep["nan%"] > 50,           "mostly NaN -> drop",
                   np.where(rep["nan%"] > 5,           "check", "ok")))
print(rep.to_string())

base = int(complete.sum())
print(f"\ncomplete rows: {base}   (which column is gating?)")
for c in ALL:
    n = int(R[[x for x in ALL if x != c]].notna().all(axis=1).sum())
    if n > base * 1.2:
        print(f"  dropping {c:<24} -> {n:>6}  (+{n-base})")


In [ ]:
CANDIDATES = ["5min", "10min", "15min"]
LAGS       = [1, 2, 3, 6, 12, 21, 24, 30, 40, 50, 60]

def acf_table(frame, title):
    print(f"\n--- {title} ---")
    print(f"{'variable':<24}" + "".join(f"{l:>7}" for l in LAGS))
    for c in frame.columns:
        s = frame[c]                            # keep NaN -> lags stay time-aligned
        if s.std() < 1e-9:
            print(f"{c:<24}   (constant)"); continue
        print(f"{c:<24}" + "".join(f"{s.autocorr(l):7.2f}" for l in LAGS))

for f in CANDIDATES:
    Rf = sw.resample(d, flows, f)
    acf_table(Rf,        f"{f} LEVELS")
    acf_table(Rf.diff(), f"{f} DIFFERENCES")


In [ ]:
MAXLAG = 360
MINS   = int(pd.Timedelta(FREQ) / pd.Timedelta("1min"))

# tau 
# how long until a variable stops resembling its own recent past. 
# Measured as the first lag where its autocorrelation drops below 1/e ≈ 0.37. 
# Units are steps, so at FREQ = "10min" a tau of 6 means one hour.

# N_eff 
# how many genuinely independent observations you have, which is not the number of rows.

def tau_1e(s):
    for k in range(1, MAXLAG + 1):
        if s.autocorr(k) < np.exp(-1):
            return k
    return None

D = R.diff()
print(f"{'variable':<24}{'tau':>8}{'N_eff':>9}{'move%':>8}")
for c in ALL:
    s = D[c]
    if s.notna().sum() < 500 or s.std() < 1e-9:
        print(f"{c:<24}   (unusable)"); continue
    t, n = tau_1e(s), s.notna().sum()
    print(f"{c:<24}{(str(t) if t else f'>{MAXLAG}'):>8}"
          f"{(n/(2*t) if t else n/(2*MAXLAG)):9.0f}{(s.abs() > 1e-9).mean()*100:8.1f}")

run  = (complete != complete.shift()).cumsum()
gaps = complete[~complete].groupby(run[~complete]).size()
print(f"\ncomplete rows : {int(complete.sum())} of {len(R)} ({complete.mean():.1%})")
print(f"gaps          : {len(gaps)} blocks | median {gaps.median():.0f} steps "
      f"({gaps.median()*MINS:.0f} min) | max {gaps.max():.0f} steps "
      f"({gaps.max()*MINS/1440:.1f} days)")


In [ ]:
def rank_report(frame, cols, name):
    x = frame[cols].replace([np.inf, -np.inf], np.nan).dropna()
    x = x.loc[:, x.std() > 1e-9]
    if len(x) < 100 or x.shape[1] < 2:
        print(f"\n{name}: too little usable data {x.shape}"); return
    z  = (x - x.mean()) / x.std()
    ev = np.linalg.svd(z.values / np.sqrt(len(z)), compute_uv=False) ** 2; ev /= ev.sum()
    cc = z.corr().abs(); np.fill_diagonal(cc.values, 0)
    print(f"\n{name}: {z.shape[1]} vars, {len(z)} rows")
    print(f"  95% needs : {int((ev.cumsum() < .95).sum() + 1)} of {z.shape[1]}")
    print(f"  max |corr|: {cc.values.max():.4f}")
    top = (cc.unstack().sort_values(ascending=False).iloc[::2].head(8)
             .rename("abs_corr").reset_index())
    top.columns = ["a", "b", "abs_corr"]
    print(top.to_string(index=False))

rank_report(C, list(C.columns), "analysis frame (differenced)")

# the same blocks in levels, for comparison
rank_report(R, [c for c in sw.OILC  if c in R], "log OIL (levels)")
rank_report(R, [c for c in sw.OXYC  if c in R], "log OXY (levels)")
rank_report(R, [c for c in sw.TEMPC if c in R], "temperatures (levels)")


In [ ]:
# ---- config ---------------------------------------------------------------

DATA  = Path("data/20250710_0000-20260729_0000.parquet")
START = "2025-07-15"
END   = "2025-10-12"

COLS     = ["ARCH #3", "ARCH #4", "TMS_nox_clean"]      # any two raw columns
FRQ      = "10min"
SHOW_RAW = True                          # faint 30s trace behind the resampled

HOURS_D = 24
T0      = "2025-09-22 00:00"                       # ~120 raw samples
t0, t1  = pd.Timestamp(T0), pd.Timestamp(T0) + pd.Timedelta(hours=HOURS_D)

# ---------------------------------------------------------------------------

from pcmci_sweep import (DATA, START, END, TARGET, TARGET_CLEAN,
                         NOX_MIN_VALID, NOX_PAD)      # stay in sync with the script

LOG_FLOWS = True          # match the pipeline (it logs OIL/OXY before differencing)
_XC = globals().setdefault("_XC", {})

def raw_cols(cols):
    cols = list(cols)
    src  = {TARGET if c == TARGET_CLEAN else c for c in cols}   # translate derived
    key  = (tuple(sorted(src)), LOG_FLOWS)
    if key not in _XC:
        need = list(dict.fromkeys(["Date", "burner_cleaning"] + sorted(src)))
        x = pd.read_parquet(DATA, columns=need)
        x["Date"] = pd.to_datetime(x["Date"])
        x = x.set_index("Date").sort_index().loc[START:]
        x = x[x.index < END]

        fl = [c for c in x.columns if c.startswith(("OIL ", "OXY "))]
        x[fl] = x[fl].where(x[fl] > 0)                          # impossible -> NaN
        wx = [c for c in x.columns if c.startswith("WEATHER_")]
        x[wx] = x[wx].where(x[wx] > -900)                       # sentinels -> NaN

        if TARGET in x.columns:                                 # build TMS_nox_clean
            step = x.index.to_series().diff().median()
            k    = max(1, int(round(pd.Timedelta(NOX_PAD) / step)))
            bad  = x[TARGET].isna() | (x[TARGET] <= NOX_MIN_VALID)
            bad  = bad.rolling(2 * k + 1, center=True, min_periods=1).max().astype(bool)
            x[TARGET_CLEAN] = x[TARGET].mask(bad)

        if LOG_FLOWS and fl:
            x[fl] = np.log(x[fl])
        _XC[key] = x
    return _XC[key]


def resample_cols(x, cols, freq):
    r = x[cols].resample(freq).mean()
    if "burner_cleaning" in x.columns:                 # same blanking as the pipeline
        bc = (x["burner_cleaning"].astype(bool).resample(freq).max()
                .astype(bool).reindex(r.index, fill_value=False))
        r.loc[bc.to_numpy()] = np.nan
    return r

x  = raw_cols(COLS)
w  = resample_cols(x, COLS, FRQ)
dw = w.diff()


PALETTE = ["#2a78d6",   # blue
           "#eb6834",   # orange
           "#009e73",   # green
           "#cc79a7",   # pink
           "#56b4e9",   # sky
           "#7a7a7a"]   # grey
assert len(COLS) <= len(PALETTE), f"add more colours for {len(COLS)} columns"


fig, axes = plt.subplots(len(COLS), 1, figsize=(13, 2.6 * len(COLS)),
                         sharex=True)                    # NOT sharey
for a, c, col in zip(np.atleast_1d(axes), COLS, PALETTE):
    a.plot(dw.loc[t0:t1, c], color=col, lw=1.0, marker="o", ms=3)
    a.axhline(0, color="#999", lw=1)
    a.set_ylabel(f"Δ {c}")
    a.set_title(f"{c}   lag-1 r = {dw[c].autocorr(1):+.3f}", fontsize=9, loc="left")
    a.grid(color="#eee", lw=0.8); a.set_axisbelow(True)
    for s in ("top", "right"): a.spines[s].set_visible(False)
plt.tight_layout(); plt.show()




In [ ]:
PAIRS   = [(TARGET_CLEAN, "ARCH #4"), (TARGET_CLEAN, "ARCH #3")]
LAG     = 0          # steps of FRQ; 0 = contemporaneous, 1 = cause leads by one bin
W_HOURS = 12
ON_DIFF = True       # match the analysis (it runs on differences)

need = sorted({c for p in PAIRS for c in p})
x    = raw_cols(need)
w    = resample_cols(x, need, FRQ)
f    = w.diff() if ON_DIFF else w

W  = int(pd.Timedelta(hours=W_HOURS) / pd.Timedelta(FRQ))
rc = pd.DataFrame({
    f"{a} →{LAG}→ {b}": f[a].shift(LAG).rolling(W, min_periods=int(W * 0.9)).corr(f[b])
    for a, b in PAIRS
})
keys = list(rc.columns)

print(f"full-window correlation ({'Δ' if ON_DIFF else 'levels'} @ {FRQ}, lag {LAG}):")
for (a, b), k in zip(PAIRS, keys):
    print(f"  {k:<34} r = {f[a].shift(LAG).corr(f[b]):+.3f}")

# ---- where do they differ most? ------------------------------------------
rc["contrast"] = rc[keys[0]] - rc[keys[1]]
best = rc.dropna().reindex(rc["contrast"].abs().sort_values(ascending=False).index).head(100)
print(f"\ntop {W_HOURS}h windows by |contrast|:")
for ts, row in best.iterrows():
    t0w = ts - pd.Timedelta(FRQ) * (W - 1)
    print(f'  T0 = "{t0w:%Y-%m-%d %H:%M}"   ' +
          "   ".join(f"{k.split(' →')[0]} {row[k]:+.2f}" for k in keys) +
          f"   Δ {row['contrast']:+.2f}")




In [ ]:
RAW_DIFF = "same_span"     # "same_span" -> raw diffed over FRQ (comparable scale)
                           # "step"      -> raw diffed one 30s step (much smaller)
HOURS = 120
step = x.index.to_series().diff().median()
k    = max(1, int(round(pd.Timedelta(FRQ) / step))) 
t0, t1 = pd.Timestamp(T0), pd.Timestamp(T0) + pd.Timedelta(hours=HOURS)
C1, C2 = "#2a78d6", "#eb6834"

def style(a, ylab, title=None):
    a.grid(color="#eee", lw=0.8); a.set_axisbelow(True)
    for s in ("top", "right"): a.spines[s].set_visible(False)
    a.set_ylabel(ylab)
    if title: a.set_title(title, loc="left")

# ---- figure 1: levels -----------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 5))
for c, col in zip(COLS, (C1, C2)):
    #if SHOW_RAW:
    #    ax.plot(x.loc[t0:t1, c], color=col, lw=0.6, alpha=0.30)
    ax.plot(w.loc[t0:t1, c], color=col, lw=1.6, marker="o", ms=4,
            label=f"{c}  @{FRQ}")
style(ax, "raw", f"raw — {t0:%Y-%m-%d %H:%M} +{HOURS}h"
                   f"{'   (faint = raw 30s)' if SHOW_RAW else ''}")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

# ---- figure 2: differences ------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 5))
for c, col in zip(COLS, (C1, C2)):
    if SHOW_RAW:
        faint = x[c].diff(k if RAW_DIFF == "same_span" else 1)
        ax.plot(faint.loc[t0:t1], color=col, lw=0.6, alpha=0.30)
    ax.plot(dw.loc[t0:t1, c], color=col, lw=1.4, marker="o", ms=4,
            label=f"Δ{c}  @{FRQ}")
ax.axhline(0, color="#999", lw=1)
lab = f"raw Δ over {FRQ}" if RAW_DIFF == "same_span" else f"raw Δ per {step}"
style(ax, f"Δ per {FRQ}", f"differences — {t0:%Y-%m-%d %H:%M} +{HOURS}h"
                          f"{f'   (faint = {lab})' if SHOW_RAW else ''}")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
need = ["ARCH #3", "ARCH #4", TARGET_CLEAN]
x = raw_cols(need); w = resample_cols(x, need, FRQ); f = w.diff()

a3, a4, nox = f["ARCH #3"], f["ARCH #4"], f[TARGET_CLEAN]
print(f"Δ corr  ARCH#3 ~ ARCH#4 : {a3.corr(a4):+.3f}\n")

common = (a3 + a4) / 2          # what they agree on
resid  =  a3 - a4               # what they disagree on
print(f"{'lag':>4}{'NOx ~ common':>15}{'NOx ~ residual':>17}")
for lag in (0, 1, 2, 3):
    print(f"{lag:>4}{nox.corr(common.shift(lag)):15.3f}"
          f"{nox.corr(resid.shift(lag)):17.3f}")
print(f"\nsd(common) {common.std():.4f}   sd(residual) {resid.std():.4f}")


In [ ]:
COMP_SRC = ["ARCH #3", "ARCH #4",
            "TEMP #08", "TEMP #09", "TEMP #10", "MELTER BT #11"]

need = COMP_SRC + [TARGET_CLEAN]
x = raw_cols(need); w = resample_cols(x, need, FRQ); f = w.diff()
print(f"{'member':<18}{'lag0':>9}{'lag1':>9}{'lag2':>9}")
for c in COMP_SRC:
    print(f"{c:<18}" + "".join(f"{f[c].shift(l).corr(f[TARGET_CLEAN]):9.3f}"
                               for l in (0, 1, 2)))
